# CatBoost: From First Principles

CatBoost is a gradient-boosting library for decision-tree ensembles. Its design is notable for handling categorical features through ordered target statistics and for reducing a form of target leakage called prediction shift through ordered boosting.

CatBoost also commonly builds symmetric (oblivious) trees, where every node at a given depth uses the same split condition. These are distinct ideas: categorical processing, boosting mode, and tree shape solve different problems. Exact behavior depends on configuration and task.

## Why CatBoost Is Useful

Many tabular datasets contain categorical features such as product, region, device type, or occupation. One-hot encoding can create a large sparse matrix, while naive target encoding can leak each training row's target into its own feature value and overfit rare categories.

CatBoost combines categorical-feature statistics with boosting procedures designed to reduce this leakage. Its common symmetric-tree structure can also make inference efficient. These ideas are useful when categories carry signal, but they do not eliminate the need for validation or thoughtful feature handling.

CatBoost is not automatically superior to other gradient-boosting libraries. Compare models on the same data split and metrics, and account for training time, memory, and deployment requirements.

## Categorical Features and Ordered Target Statistics

A categorical feature such as product type or city has labels rather than a natural numeric scale. Treating arbitrary category codes as numbers can invent an order that does not exist, while one-hot encoding can create many columns for high-cardinality features.

CatBoost can derive numerical statistics from categories and category combinations. A simplified target statistic for category $c$ uses only earlier observations in a permutation:

$$
\mathrm{TS}(c) = \frac{\sum_{j\prec i,\,x_j=c} y_j + aP}{\#\{j\prec i:x_j=c\}+a}
$$

Here $j\prec i$ means observation $j$ appears before $i$ in the permutation, $P$ is a prior, and $a$ controls the prior's influence. Because observation $i$'s own target and later targets are excluded when creating its training statistic, the encoding avoids the direct self-target leakage that would occur if each row were encoded using the full-data category mean.

This formula is a simplified illustration. CatBoost supports multiple categorical statistics, target types, permutations, and combinations; exact behavior depends on configuration.

## Example: A Training-Row Category Statistic

Suppose a category `blue` appears in earlier training rows with binary targets 1 and 0. Let the prior be $P=0.5$ and prior strength be $a=1$. For a later row whose category is `blue`, the simplified ordered statistic is:

$$
\mathrm{TS}(\text{blue})=\frac{1+0+1(0.5)}{2+1}=0.5
$$

The row's own target is not included, and neither are targets later in the permutation. A full-data category mean could include the row's target and make rare categories look artificially predictive; the ordered calculation avoids that direct self-inclusion during training.

At prediction time, unseen or previously observed categories are handled using statistics derived from the training data and configured priors. The exact statistic and fallback behavior depend on CatBoost's selected categorical features and settings.

## Ordered Boosting and Prediction Shift

Target statistics address leakage while constructing categorical features. Ordered boosting addresses a related but separate issue in the boosting process: **prediction shift**. In ordinary boosting, a training row's current prediction comes from a model fitted using that same row. Its loss gradient can therefore be statistically different from the gradient expected on unseen data.

The ordered idea uses permutations of the training data. For a row at a position in a permutation, its training prediction is formed using models trained only on preceding rows in that permutation. This makes the prediction used to compute its gradient more like a prediction for a new observation, reducing the direct influence of its own target.

Ordered target statistics and ordered boosting have related motivation but act at different stages: one constructs category-derived features, the other changes how training predictions and gradients are obtained. CatBoost also provides a `Plain` boosting mode; the available and default modes depend on task, device, and configuration.

## Ordered Boosting with Prefix-Trained Learners

Use squared-error loss, a fixed starting score $F_0=0$, and this permutation of four rows: $A,B,C,D$, with targets $y=(0,0,1,0)$. To isolate the ordering rule, let each prefix model be a one-leaf regression tree fit to the residuals of only the rows before the current row. This gives a literal prefix-trained boosting example with a deliberately simple weak learner; it is not a dump of CatBoost library predictions.

In the first round, the residuals from $F_0$ are $y-F_0=(0,0,1,0)$. Each row's training prediction is made by a model that sees only the earlier prefix:

| Row | Earlier prefix used to fit its model | One-leaf update for that row | Ordered prediction |
|---|---|---:|---:|
| $A$ | Empty; use fixed starting score | None | $0$ |
| $B$ | $A$ with residual $0$ | Mean residual $0$ | $0$ |
| $C$ | $A,B$ with residuals $0,0$ | Mean residual $0$ | $0$ |
| $D$ | $A,B,C$ with residuals $0,0,1$ | Mean residual $1/3$ | $1/3$ |

At the next gradient calculation, the ordered training predictions are $(0,0,0,1/3)$, so the negative gradients for squared error are $y-F=(0,0,1,-1/3)$. None of these predictions used that row's own target or any later target.

For comparison, a plain one-leaf learner fit to all four first-round residuals would predict their global mean $1/4$ for every training row. Its residuals would be $(-1/4,-1/4,3/4,-1/4)$, because each row, including the row being predicted, helped determine that shared mean. Ordered boosting uses prefix-trained predictions to reduce this self-influence (prediction shift). Production CatBoost uses permutation-specific boosting models and its full tree/categorical-statistic machinery, so its exact outputs are more involved than this constant-tree demonstration.

## Symmetric Trees and the Boosting Model

Like other gradient-boosting systems, CatBoost adds trees sequentially so later trees improve the current model under a chosen loss. Its common default tree structure is symmetric, also called an oblivious tree: every node at the same depth uses the same split condition.

In a full symmetric tree of depth $d$, each row follows one of $2^d$ paths. Reusing each depth's split condition makes the structure regular and can make prediction efficient, but it constrains which partitions the tree can represent compared with an unrestricted tree.

CatBoost exposes tree-growth choices, and the symmetric structure should not be confused with ordered boosting. Tree shape controls how each learner partitions feature space; boosting mode controls how the ensemble is trained.

## Important Parameters and Workflow

| Parameter or choice | Role |
|---|---|
| `loss_function` | Defines the training objective for regression, classification, or another supported task |
| `iterations` | Maximum number of boosting rounds |
| `learning_rate` | Shrinks each tree's contribution; smaller values often need more rounds |
| `depth` | Controls the depth and complexity of each tree |
| `l2_leaf_reg` | L2 regularization on leaf values |
| `cat_features` | Identifies categorical columns when the input format does not already communicate their types |
| `eval_metric` | Chooses the metric monitored for validation or early stopping |
| `boosting_type` | Selects a boosting mode such as `Ordered` or `Plain` where supported |

Split data before fitting any learned preprocessing. Pass categorical columns using the representation expected by the API, and do not convert category labels to numeric codes that imply a false order. Use a validation set to tune parameters and early stopping; reserve the test set for final assessment. Parameter names and available modes vary somewhat by interface and version.

## When to Use CatBoost

CatBoost is a good candidate for structured data with useful categorical features, especially when manual one-hot encoding would create many columns. It can reduce preprocessing effort and combines categorical handling with boosted trees for nonlinear relationships and interactions.

Its specialized handling does not remove all risks. Rare categories can have unstable target statistics; symmetric trees constrain the set of partitions; and ordered procedures can add computation. CatBoost also cannot fix poor evaluation design, temporal leakage, unrepresentative data, or mislabeled examples.

Use a validation strategy that matches the task, compare CatBoost with suitable baselines, inspect errors and subgroup performance, and use calibrated probabilities if downstream decisions require them. Choose the model based on measured quality, reliability, training cost, and deployment needs rather than categorical-feature support alone.